# Repository and AVeriTeC Setup

This notebook prepares the reproducible development environment used by the
evidence-retrieval experiments.

It will:

1. locate or clone the project repository;
2. create the local project/data structure used by the experiment notebooks;
3. install the project and data-download dependencies;
4. download the AVeriTeC train and development annotations at the exact revision
   used by the dissertation experiments;
5. optionally download and extract the full development knowledge store;
6. record the dataset revision and local path in a manifest and `.env` file.

The AVeriTeC development knowledge store is large, so the download/extraction
steps remain explicitly configurable.

## 1. Project Configuration

In [1]:
GITHUB_REPOSITORY = (
    "https://github.com/REALBroomFish/"
    "AutomatedFactVerification_RetrieverComparison.git"
)
GIT_BRANCH = "main"

HF_REPOSITORY = "chenxwh/AVeriTeC"

# Exact AVeriTeC revision used by the dissertation experiments
HF_REVISION = "2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031"

DOWNLOAD_FULL_DEV_KNOWLEDGE_STORE = False
EXTRACT_FULL_DEV_KNOWLEDGE_STORE = False

PROJECT_NAME = GITHUB_REPOSITORY.rstrip("/").split("/")[-1].removesuffix(".git")

print("Project:", PROJECT_NAME)
print("Pinned AVeriTeC revision:", HF_REVISION)

Project: AutomatedFactVerification_RetrieverComparison
Pinned AVeriTeC revision: 2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031


## 2. Detect if in Google Colab and prepare storage

In [2]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    WORK_ROOT = Path("/content")

    # Persistent storage for the large dataset in Colab.
    PERSISTENT_DATA_ROOT = (Path("/content/drive/MyDrive") / "MSc_Dissertation" / "datasets")
else:
    WORK_ROOT = Path.cwd().resolve()
    PERSISTENT_DATA_ROOT = None

print("Running in Colab:", IN_COLAB)
print("Working directory:", WORK_ROOT)

Running in Colab: False
Working directory: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\notebooks


## 3. Clone or locate repository

In [3]:
import os
import subprocess
from pathlib import Path


def run_command(command: list[str], *, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and show stdout/stderr if it fails."""
    print("$", " ".join(command))

    result = subprocess.run(command, cwd=cwd, check=False, text=True, capture_output=True)

    if result.stdout:
        print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}:\n" f"{result.stderr}")

    return result


def find_git_repository(start: Path) -> Path | None:
    """Search the current directory and its parents for a .git directory."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists():
            return candidate

    return None


existing_repository = find_git_repository(Path.cwd())

if existing_repository is not None:
    PROJECT_ROOT = existing_repository
    print(f"Using existing repository at:\n{PROJECT_ROOT}")
else:
    PROJECT_ROOT = WORK_ROOT / PROJECT_NAME

    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"{PROJECT_ROOT} already exists and is not empty.")

    run_command(["git", "clone", "--branch", GIT_BRANCH, GITHUB_REPOSITORY, str(PROJECT_ROOT)])

os.chdir(PROJECT_ROOT)

# keep local data in one canonical repo level location
if IN_COLAB:
    DATA_ROOT = PERSISTENT_DATA_ROOT
else:
    DATA_ROOT = PROJECT_ROOT / ".local_data"

DATA_ROOT.mkdir(parents=True, exist_ok=True)

run_command(["git", "status", "--short"], cwd=PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

Using existing repository at:
C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison
$ git status --short
 M notebooks/0_repository_and_data_setup.ipynb
 M notebooks/2_retrieval_evaluation.ipynb

Project root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison


## 4. Ensure Current Repository Structure

The experiment notebooks use `src/` for the package, `notebooks/` for notebook
entry points, `results/` for generated experiment outputs, and `.local_data/`
for the large local AVeriTeC files.

In [4]:
directories = [PROJECT_ROOT / "notebooks", PROJECT_ROOT / "src", PROJECT_ROOT / "data", PROJECT_ROOT / "results"]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

data_readme = PROJECT_ROOT / "data" / "README.md"

if not data_readme.exists():
    data_readme.write_text(
        "# Data\n\n Large dataset files are stored outside Git.\n"
        "The active AVeriTeC dataset root is configured through `AVERITEC_ROOT`.\n",
        encoding="utf-8")
    
gitignore_path = PROJECT_ROOT / ".gitignore"

required_gitignore_entries = [".env", ".local_data/", ".ipynb_checkpoints/", "__pycache__/", "*.py[cod]", "results/", "data/*",
                              "!data/README.md"]
existing_gitignore = (gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else "")
missing_entries = [entry for entry in required_gitignore_entries if entry not in existing_gitignore.splitlines()]

if missing_entries:
    with gitignore_path.open("a", encoding="utf-8") as file:
        if existing_gitignore and not existing_gitignore.endswith("\n"):
            file.write("\n")

        file.write("\n# Local project artefacts\n")
        file.write("\n".join(missing_entries))
        file.write("\n")

print("Repository structure checked.")

Repository structure checked.


## 5. install dependencies


In [5]:
# Use the Python interpreter attached to this notebook.
run_command([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_xet", "pandas", "python-dotenv"], cwd=PROJECT_ROOT)

# Install the project itself in editable mode.
pyproject_path = PROJECT_ROOT / "pyproject.toml"

if pyproject_path.exists():
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)], cwd=PROJECT_ROOT)
    print("Project installed in editable mode.")
else:
    print("No pyproject.toml found; skipped editable project installation")

$ c:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.venv\Scripts\python.exe -m pip install -q huggingface_hub hf_xet pandas python-dotenv

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

$ c:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.venv\Scripts\python.exe -m pip install -q -e C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip

Project installed in editable mode.


## 6. Verify the Pinned Hugging Face Revision

In [6]:
from huggingface_hub import HfApi, hf_hub_download

hf_api = HfApi()
repository_info = hf_api.model_info(HF_REPOSITORY, revision=HF_REVISION)

if repository_info.sha != HF_REVISION:
    raise RuntimeError(f"The resolved Hugging Face revision does not match the pinned revision: {repository_info.sha}")

print("Hugging Face repository:", HF_REPOSITORY)
print("Verified revision:", repository_info.sha)

c:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face repository: chenxwh/AVeriTeC
Verified revision: 2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031


## 7. Download annotation files

In [7]:
DATASET_ROOT = (DATA_ROOT / "averitec" / HF_REVISION[:12])
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

ANNOTATION_FILES = ["data/train.json", "data/dev.json"]
downloaded_files: dict[str, Path] = {}

for filename in ANNOTATION_FILES:
    local_path = Path(hf_hub_download(repo_id=HF_REPOSITORY, filename=filename, revision=HF_REVISION, local_dir=DATASET_ROOT))

    downloaded_files[filename] = local_path
    print(f"Downloaded {filename} -> {local_path}")

Downloaded data/train.json -> C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\data\train.json
Downloaded data/dev.json -> C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\data\dev.json


## 8. Validate annotations

In [8]:
import json

TRAIN_PATH = downloaded_files["data/train.json"]
DEV_PATH = downloaded_files["data/dev.json"]


def load_json(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(path)

    with path.open("r", encoding="utf-8") as file:
        records = json.load(file)

    if not isinstance(records, list):
        raise TypeError(f"Expected a list of records in {path}, " f"received {type(records).__name__}.")

    return records


train_records = load_json(TRAIN_PATH)
dev_records = load_json(DEV_PATH)
assert len(train_records) == 3068
assert len(dev_records) == 500

print(f"Training records: {len(train_records):,}")
print(f"Development records: {len(dev_records):,}")

Training records: 3,068
Development records: 500


### 8.1. Small annotation inspection

In [9]:
first_record = dev_records[0]

expected_fields = {"claim", "label", "justification", "questions"}

missing_fields = expected_fields.difference(first_record)

if missing_fields:
    raise KeyError(f"The first development record is missing: {missing_fields}")

print("Available fields:")
print(sorted(first_record.keys()))

print("\nClaim:")
print(first_record["claim"])

print("\nLabel:")
print(first_record["label"])

print("\nNumber of annotated questions:")
print(len(first_record["questions"]))

Available fields:
['cached_original_claim_url', 'claim', 'claim_date', 'claim_types', 'fact_checking_article', 'fact_checking_strategies', 'justification', 'label', 'location_ISO_code', 'original_claim_url', 'questions', 'reporting_source', 'required_reannotation', 'speaker']

Claim:
In a letter to Steve Jobs, Sean Connery refused to appear in an apple commercial.

Label:
Refuted

Number of annotated questions:
2


## 9. Record a dataset manifest

In [10]:
from datetime import datetime, timezone

manifest = {
    "dataset": "AVeriTeC",
    "huggingface_repository": HF_REPOSITORY,
    "revision": HF_REVISION,
    "downloaded_at_utc": datetime.now(timezone.utc).isoformat(),
    "files": {filename: {"path": str(path.relative_to(DATASET_ROOT)), "size_bytes": path.stat().st_size} for filename, path in downloaded_files.items()},
}

manifest_path = DATASET_ROOT / "dataset_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(manifest_path.read_text(encoding="utf-8"))

{
  "dataset": "AVeriTeC",
  "huggingface_repository": "chenxwh/AVeriTeC",
  "revision": "2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031",
  "downloaded_at_utc": "2026-08-27T22:59:23.330401+00:00",
  "files": {
    "data/train.json": {
      "path": "data\\train.json",
      "size_bytes": 10184813
    },
    "data/dev.json": {
      "path": "data\\dev.json",
      "size_bytes": 1785475
    }
  }
}


## 10. Register the Local Data Location

In [11]:
from dotenv import set_key

env_path = PROJECT_ROOT / ".env"

# create the local environment file if it does not already exist
env_path.touch(exist_ok=True)

# update only the AVeriTeC values without replacing any other local settings
set_key(str(env_path), "AVERITEC_ROOT", str(DATASET_ROOT))
set_key(str(env_path), "AVERITEC_REVISION", HF_REVISION)

# also register the values in the current notebook process
os.environ["AVERITEC_ROOT"] = str(DATASET_ROOT)
os.environ["AVERITEC_REVISION"] = HF_REVISION

print("Updated local configuration:", env_path)
print("AVERITEC_ROOT:", os.environ["AVERITEC_ROOT"])
print("AVERITEC_REVISION:", os.environ["AVERITEC_REVISION"])

Updated local configuration: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.env
AVERITEC_ROOT: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a
AVERITEC_REVISION: 2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031


## 11. Download the Full Development Knowledge Store

At the pinned AVeriTeC revision the development knowledge-store archive is
approximately 10.75 GiB compressed and roughly 34 GiB after extraction.
The retrieval experiment requires the extracted `output_dev` directory.

In [12]:
import shutil
import zipfile


KNOWLEDGE_STORE_FILENAME = "data_store/knowledge_store/dev_knowledge_store.zip"
KNOWLEDGE_STORE_PARENT = DATASET_ROOT / "knowledge_store" / "dev"
KNOWLEDGE_STORE_DIRECTORY = KNOWLEDGE_STORE_PARENT / "output_dev"

if DOWNLOAD_FULL_DEV_KNOWLEDGE_STORE:
    print("Free persistent storage before download:", f"{shutil.disk_usage(DATA_ROOT).free / (1024**3):.2f} GiB")

    knowledge_store_zip = Path(hf_hub_download(repo_id=HF_REPOSITORY, filename=KNOWLEDGE_STORE_FILENAME, revision=HF_REVISION, 
                                               local_dir=DATASET_ROOT))

    print("Knowledge-store archive:", knowledge_store_zip)
    print("Compressed archive size:", f"{knowledge_store_zip.stat().st_size / (1024**3):.2f} GiB")

    if EXTRACT_FULL_DEV_KNOWLEDGE_STORE:
        # avoid extracting ~34 GiB archive again when complete dev knowledge store is already present
        existing_claim_files = []
        if KNOWLEDGE_STORE_DIRECTORY.exists():
            existing_claim_files = list(KNOWLEDGE_STORE_DIRECTORY.glob("*.json"))

        if len(existing_claim_files) == len(dev_records):
            print("Development knowledge store is already fully extracted.")
        else:
            with zipfile.ZipFile(knowledge_store_zip) as archive:
                uncompressed_bytes = sum(member.file_size for member in archive.infolist())
                free_bytes = shutil.disk_usage(DATA_ROOT).free

                print("Expected extracted size:", f"{uncompressed_bytes / (1024**3):.2f} GiB")
                print("Currently free:", f"{free_bytes / (1024**3):.2f} GiB")

                # retain a 10% safety margin while extracting
                required_bytes = int(uncompressed_bytes * 1.10)

                if free_bytes < required_bytes:
                    raise RuntimeError("Insufficient free storage to extract the development knowledge store safely.")

                KNOWLEDGE_STORE_PARENT.mkdir(parents=True, exist_ok=True)
                archive.extractall(KNOWLEDGE_STORE_PARENT)

            # expected structure is:
            #
            # knowledge_store/
            # └── dev/
            #     └── output_dev/
            #         ├── 0.json
            #         ├── 1.json
            #         └── ...
            #
            # resolve output_dev defensively in case the archive contains an additional directory level
            if not KNOWLEDGE_STORE_DIRECTORY.exists():
                matches = [path for path in KNOWLEDGE_STORE_PARENT.rglob("output_dev") if path.is_dir()]

                if len(matches) == 1:
                    KNOWLEDGE_STORE_DIRECTORY = (matches[0])
                else:
                    raise FileNotFoundError("Extraction completed, but the expected `output_dev` directory could not be resolved.")


        # validate extracted knowledge store regardless of whether it was newly extracted or already present
        claim_files = list(KNOWLEDGE_STORE_DIRECTORY.glob("*.json"))

        if len(claim_files) != len(dev_records):
            raise RuntimeError("Knowledge-store claim count does not match the development split: "
                               f"{len(claim_files)} files for {len(dev_records)} claims.")

        print("Development knowledge store:", KNOWLEDGE_STORE_DIRECTORY)
        print("Knowledge-store claim files:", len(claim_files))

else:
    print("Full development knowledge store not downloaded. Set DOWNLOAD_FULL_DEV_KNOWLEDGE_STORE=True to download it.")

Full development knowledge store not downloaded. Set DOWNLOAD_FULL_DEV_KNOWLEDGE_STORE=True to download it.


## 12. Final Setup check

In [13]:
print("Repository")
print("----------")
print("Root:", PROJECT_ROOT)


print("\nDataset")
print("-------")
print("Root:", DATASET_ROOT)
print("Revision:", HF_REVISION)
print("Train exists:", TRAIN_PATH.exists())
print("Dev exists:", DEV_PATH.exists())
print("Training claims:", len(train_records))
print("Development claims:", len(dev_records))


print("\nKnowledge store")
print("---------------")
print("Expected directory:", KNOWLEDGE_STORE_DIRECTORY)
knowledge_store_exists = KNOWLEDGE_STORE_DIRECTORY.exists()

print("Exists:", knowledge_store_exists)

if knowledge_store_exists:
    knowledge_store_claim_files = list(KNOWLEDGE_STORE_DIRECTORY.glob("*.json"))

    print("Claim files:", len(knowledge_store_claim_files))

    if (len(knowledge_store_claim_files)!= len(dev_records)):
        raise RuntimeError("Knowledge-store claim count does not match the development split.")

print("\nEnvironment")
print("-----------")
print("AVERITEC_ROOT:", os.environ.get("AVERITEC_ROOT"))
print("AVERITEC_REVISION:", os.environ.get("AVERITEC_REVISION"))

print("\nSample claim")
print("------------")
print(dev_records[0]["claim"])


# Final assertions for the experimental dataset
assert TRAIN_PATH.exists(), "AVeriTeC train annotations are missing."
assert DEV_PATH.exists(), "AVeriTeC development annotations are missing."

assert len(train_records) == 3068, "Unexpected number of AVeriTeC training records."
assert len(dev_records) == 500, "Unexpected number of AVeriTeC development records."

assert os.environ.get("AVERITEC_REVISION") == HF_REVISION, "AVERITEC_REVISION does not match the pinned dataset revision."

print("\nAnnotation setup: PASS")

if knowledge_store_exists:
    assert len(knowledge_store_claim_files) == 500, "Development knowledge store does not contain all 500 claims."
    print("Retrieval dataset setup: PASS")
else:
    print("Retrieval dataset setup: INCOMPLETE")
    print("The development knowledge store is required before running the retrieval experiment.")

print("\nGit status")
print("----------")

run_command(["git", "status", "--short"], cwd=PROJECT_ROOT)

Repository
----------
Root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison

Dataset
-------
Root: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a
Revision: 2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031
Train exists: True
Dev exists: True
Training claims: 3068
Development claims: 500

Knowledge store
---------------
Expected directory: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a\knowledge_store\dev\output_dev
Exists: True
Claim files: 500

Environment
-----------
AVERITEC_ROOT: C:\Users\conno_i4oarur.MACPEPINO\OneDrive\Documents\GitHub\AutomatedFactVerification_RetrieverComparison\.local_data\averitec\2ca9dee23a2a
AVERITEC_REVISION: 2ca9dee23a2a6fa64c5bd918e0cd28ed0aa09031

Sample claim
------------
In a letter to Steve Jobs, Sean Connery refused to ap

CompletedProcess(args=['git', 'status', '--short'], returncode=0, stdout=' M notebooks/0_repository_and_data_setup.ipynb\n M notebooks/2_retrieval_evaluation.ipynb\n M src/factverify.egg-info/PKG-INFO\n M src/factverify.egg-info/requires.txt\n', stderr='')